# Demo 03 : Router

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.5", "langchain-core==1.6.10", "langchain-openai==1.7.1", "langgraph==1.2.15", "langgraph-checkpoint-sqlite==3.1.2", "deepagents==0.7.24", "langsmith==0.14.8", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    THINKING_OFF = {"reasoning": {"effort": "none"}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# New OpenRouter accounts may send 20 requests a minute per model.
# One shared limiter keeps parallel agents under it ; raise COURSE_RPS when the cap lifts
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "0.3")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    kw.setdefault("temperature", 0)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


We use the made-up Veenhaven Bus records. We count every attempted model call, input and output tokens, and elapsed seconds, including the shared rate limiter.

In [2]:
import json
import operator
import re
import sqlite3
import threading
import time
from pathlib import Path
from typing import Annotated, Literal
from typing_extensions import NotRequired, TypedDict

from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain.tools import ToolRuntime, tool
from langchain_core.callbacks import BaseCallbackHandler
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.types import Command, Send
from pydantic import BaseModel, Field


class Counter(BaseCallbackHandler):
    def __init__(self):
        self.lock = threading.Lock()
        self.reset()

    def reset(self):
        self.calls = self.input = self.output = self.invalid = self.errors = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        with self.lock:
            self.calls += 1

    def on_llm_end(self, response, **kwargs):
        with self.lock:
            for generations in response.generations:
                for generation in generations:
                    message = getattr(generation, "message", None)
                    usage = getattr(message, "usage_metadata", None) or {}
                    self.input += usage.get("input_tokens", 0)
                    self.output += usage.get("output_tokens", 0)
                    self.invalid += len(getattr(message, "invalid_tool_calls", []) or [])

    def on_llm_error(self, error, **kwargs):
        with self.lock:
            self.errors += 1


counter = Counter()
llm = make_llm(callbacks=[counter], max_tokens=1200, max_retries=0)
metrics = []


def measured(label, operation):
    counter.reset()
    start = time.perf_counter()
    try:
        return operation()
    finally:
        row = dict(pattern=label, calls=counter.calls, input_tokens=counter.input,
                   output_tokens=counter.output, seconds=round(time.perf_counter() - start, 1),
                   malformed=counter.invalid, errors=counter.errors)
        metrics.append(row)
        print("metrics :", json.dumps(row))


DB = Path(data("demo-world/service.db")).resolve()
MANUALS = Path(data_folder("demo-world/manuals"))
PAGES = {p.name: p.read_text(encoding="utf-8") for p in sorted(MANUALS.glob("*.md"))}


@tool
def part_stock(part: str) -> str:
    """Look up the stock, location and name of a Veenhaven Bus part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, name, stock, depot FROM parts WHERE part_no = ?",
                          (part.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Read the manual's meaning and first action for a Veenhaven Bus error code."""
    matches = [f"[{name}]\n{line}" for name, text in PAGES.items()
               for line in text.splitlines() if line.startswith("| " + error_code.upper() + " |")]
    return "\n".join(matches) or "No matching error code."


@tool
def run_sql(query: str) -> str:
    """Run a read-only SELECT. Tables: buses(fleet_no, model, depot_id, commissioned, firmware),
    depots(depot_id, name, city, contract), parts(part_no, model, name, stock, price_eur, depot),
    work_orders(wo_id, fleet_no, opened, status, error_code, summary, mechanic_id)."""
    if not query.lstrip().lower().startswith("select"):
        return "Only SELECT statements are allowed."
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        return json.dumps(db.execute(query).fetchmany(30))


@tool
def search_manuals(query: str) -> str:
    """Find relevant Veenhaven Bus manuals and bulletins with their file names."""
    words = set(query.lower().split())
    ranked = sorted(PAGES, key=lambda name: -sum(PAGES[name].lower().count(w) for w in words))[:3]
    return "\n\n".join(f"[{name}]\n{PAGES[name]}" for name in ranked)


QUESTION = ("Bus CE12-2021-0117 shows F-CE-110. How many CE-CF-02 cooling fans are in stock, "
            "at which location, and what are the manual's first action and replacement condition for this error? Be brief.")
STOCK_TASK = "How many CE-CF-02 cooling fans are in stock and at which location?"
MANUAL_TASK = "What are the meaning, first action and replacement condition for F-CE-110? Cite the manual."
LIMIT = {"recursion_limit": 24}


def ask(question):
    return {"messages": [HumanMessage(question)]}


def last_text(result):
    return str(next(m.content for m in reversed(result["messages"])
                    if isinstance(m, AIMessage) and m.content and not m.tool_calls))


def agent_by_hand(model, tools, prompt, state_schema=MessagesState):
    bound = model.bind_tools(tools)

    def model_node(state):
        return {"messages": [bound.invoke([SystemMessage(prompt), *state["messages"]])]}

    graph = StateGraph(state_schema)
    graph.add_node("model", model_node)
    graph.add_node("tools", ToolNode(tools))
    graph.add_edge(START, "model")
    graph.add_conditional_edges("model", tools_condition)
    graph.add_edge("tools", "model")
    return graph.compile()


def specialists(model):
    return {
        "parts": agent_by_hand(model, [part_stock], "Answer stock questions from the tool. Return every fact briefly."),
        "manuals": agent_by_hand(model, [fault_guidance], "Answer error-code questions from the manual tool. Cite its file."),
    }


def compose(model, question, findings):
    return model.invoke("Answer from these findings only. Preserve counts, locations and conditions.\n"
                        + question + "\n" + "\n".join(findings)).content


def completeness(answer):
    text = answer.lower()
    return {"stock": bool(re.search(r"\b(6|six)\b", text)) and "rietpoort" in text,
            "action": "fan" in text and "airflow" in text and bool(re.search(r"\b(80|eighty)\b", text))}

## a · router

We classify a stock request and dispatch it to the selected specialist. We keep the routing decision visible instead of asking the coordinator to route on every turn.

In [3]:
class Route(BaseModel):
    agent: Literal["parts", "manuals"]
    query: str


class Routes(BaseModel):
    routes: list[Route] = Field(min_length=1, max_length=2)


class RouterState(TypedDict):
    question: str
    routes: list[dict]
    answers: Annotated[list[str], operator.add]
    final: str


def router_graph(model, parallel):
    workers = specialists(model)

    def classify(state):
        routes = model.with_structured_output(Routes).invoke(
            "Choose the required specialists. parts handles stock, manuals handles error codes. "
            "At most one question per specialist.\n" + state["question"]).routes
        print("routes :", [(r.agent, r.query) for r in routes])
        return {"routes": [r.model_dump() for r in routes]}

    def dispatch(state):
        return [Send(r["agent"], {"question": r["query"]}) for r in state["routes"]]

    def specialist(name):
        return lambda s: {"answers": [name + " : " + last_text(workers[name].invoke(ask(s["question"]), LIMIT))]}

    def sequential_dispatch(state):
        return {"answers": [r["agent"] + " : " + last_text(workers[r["agent"]].invoke(ask(r["query"]), LIMIT))
                            for r in state["routes"]]}

    graph = StateGraph(RouterState)
    graph.add_node("classify", classify)
    graph.add_node("synthesise", lambda s: {"final": compose(model, s["question"], s["answers"])})
    graph.add_edge(START, "classify")
    if parallel:
        for name in workers:
            graph.add_node(name, specialist(name))
            graph.add_edge(name, "synthesise")
        graph.add_conditional_edges("classify", dispatch, ["parts", "manuals"])
    else:
        graph.add_node("dispatch", sequential_dispatch)
        graph.add_edge("classify", "dispatch")
        graph.add_edge("dispatch", "synthesise")
    graph.add_edge("synthesise", END)
    return graph.compile()


result = measured("router sequential", lambda: router_graph(llm, False).invoke(
    {"question": STOCK_TASK, "answers": []}, LIMIT))
print("answer :", result["final"])

routes : [('parts', 'How many CE-CF-02 cooling fans are in stock, and at which location?')]


metrics : {"pattern": "router sequential", "calls": 4, "input_tokens": 419, "output_tokens": 108, "seconds": 34.8, "malformed": 0, "errors": 0}
answer : There are 6 CE-CF-02 traction cooling fans in stock at Rietpoort.


## b · router-parallel

We route a request with independent stock and manual questions to both workers using Send. Each branch adds an answer through a reducer, and the synthesis sees the combined results.

In [4]:
result = measured("router parallel", lambda: router_graph(llm, True).invoke({"question": QUESTION, "answers": []}, LIMIT))
print("branches :", len(result["answers"]))
print("answer :", result["final"])
print("checks :", completeness(result["final"]))

routes : [('parts', 'How many CE-CF-02 cooling fans are in stock, and at which location(s)?'), ('manuals', "For bus CE12-2021-0117 showing error F-CE-110, what is the manual's first action and replacement condition?")]


metrics : {"pattern": "router parallel", "calls": 6, "input_tokens": 824, "output_tokens": 261, "seconds": 50.7, "malformed": 0, "errors": 0}
branches : 2
answer : There are **6** CE-CF-02 traction cooling fans in stock, all at **Rietpoort**. The manual says to **check the traction cooling fan** first and **replace it if airflow is below 80 percent**.
checks : {'stock': True, 'action': True}


We gain independent execution, but the shared limiter still spaces the requests. Parallel dispatch alone does not establish a latency improvement.

## Things to try

Optional : we can change the question, inspect each context, and compare the answer checks with the call and token counts.